In [1]:
!pip install transformers datasets seqeval -q
!pip install evaluate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

# POS Dataset (Universal Dependencies - working version)
ud_dataset = load_dataset("a3lem/universal-dependencies-parquet", "en_ewt")

# Chunking / NER Dataset (CoNLL - fixed parquet version)
conll_dataset = load_dataset("lukasgarbas/conll-03")

# NER Dataset (WikiANN)
wikiann_dataset = load_dataset("wikiann", "en")

print("UD Loaded:", ud_dataset)
print("\nCoNLL Loaded:", conll_dataset)
print("\nWikiANN Loaded:", wikiann_dataset)

README.md: 0.00B [00:00, ?B/s]

data/en_ewt/test-00000-of-00001.parquet:   0%|          | 0.00/519k [00:00<?, ?B/s]

data/en_ewt/validation-00000-of-00001.pa(…):   0%|          | 0.00/516k [00:00<?, ?B/s]

data/en_ewt/train-00000-of-00001.parquet:   0%|          | 0.00/3.61M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2077 [00:00<?, ? examples/s]

...

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

UD Loaded: DatasetDict({
    test: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 2077
    })
    validation: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 2001
    })
    train: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 12544
    })
})

CoNLL Loaded: DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

WikiANN Loaded: DatasetDict({
    vali

In [3]:
# POS labels (UD)
# The 'upos' feature in ud_dataset is a Sequence of Value (strings), not ClassLabel.
# We need to extract unique values from the dataset to get the labels.
all_upos_tags = set()
for example in ud_dataset["train"]:
    for tag in example["upos"]:
        all_upos_tags.add(tag)
ud_label_names = sorted(list(all_upos_tags))

# Chunking labels (CoNLL)
conll_chunk_labels = conll_dataset["train"].features["chunk_tags"].feature.names

# NER labels (CoNLL)
conll_ner_labels = conll_dataset["train"].features["ner_tags"].feature.names

# NER labels (WikiANN)
wikiann_ner_labels = wikiann_dataset["train"].features["ner_tags"].feature.names


print("UD POS Labels:\n", ud_label_names)
print("\nCoNLL Chunk Labels:\n", conll_chunk_labels)
print("\nCoNLL NER Labels:\n", conll_ner_labels)
print("\nWikiANN NER Labels:\n", wikiann_ner_labels)

UD POS Labels:
 ['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X', '_']

CoNLL Chunk Labels:
 ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']

CoNLL NER Labels:
 ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

WikiANN NER Labels:
 ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
# Remove '_' from POS labels
ud_label_names = [label for label in ud_label_names if label != "_"]

# Create mappings
label2id = {label: i for i, label in enumerate(ud_label_names)}
id2label = {i: label for label, i in label2id.items()}

print("label2id:\n", label2id)
print("\nid2label:\n", id2label)

label2id:
 {'ADJ': 0, 'ADP': 1, 'ADV': 2, 'AUX': 3, 'CCONJ': 4, 'DET': 5, 'INTJ': 6, 'NOUN': 7, 'NUM': 8, 'PART': 9, 'PRON': 10, 'PROPN': 11, 'PUNCT': 12, 'SCONJ': 13, 'SYM': 14, 'VERB': 15, 'X': 16}

id2label:
 {0: 'ADJ', 1: 'ADP', 2: 'ADV', 3: 'AUX', 4: 'CCONJ', 5: 'DET', 6: 'INTJ', 7: 'NOUN', 8: 'NUM', 9: 'PART', 10: 'PRON', 11: 'PROPN', 12: 'PUNCT', 13: 'SCONJ', 14: 'SYM', 15: 'VERB', 16: 'X'}


In [6]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        else:
            label = example["upos"][word_idx]

            # Ignore '_' labels
            if label == "_":
                labels.append(-100)
            else:
                labels.append(label2id[label])

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [7]:
tokenized_ud = ud_dataset.map(tokenize_and_align_labels, batched=False)

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

In [8]:
tokenized_ud.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [9]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(ud_label_names),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch"
)

In [11]:
import evaluate

metric = evaluate.load("seqeval")

In [12]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, label in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for p_, l_ in zip(pred, label):
            if l_ != -100:
                current_preds.append(id2label[p_])
                current_labels.append(id2label[l_])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

In [13]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [14]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ud["train"],
    eval_dataset=tokenized_ud["validation"],
    data_collator=data_collator,   # ✅ NEW
    compute_metrics=compute_metrics,
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.116074,0.149473,0.954547,0.960295,0.957412
2,0.065781,0.140165,0.960417,0.966442,0.963420


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171:

TrainOutput(global_step=3136, training_loss=0.15774691773920643, metrics={'train_runtime': 438.1307, 'train_samples_per_second': 57.261, 'train_steps_per_second': 7.158, 'total_flos': 575294238043056.0, 'train_loss': 0.15774691773920643, 'epoch': 2.0})

In [17]:
import torch

sentence = "John works at Google in California"

inputs = tokenizer(
    sentence.split(),
    return_tensors="pt",
    is_split_into_words=True
)

# Move inputs to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

predictions = torch.argmax(outputs.logits, dim=2)

predicted_labels = [
    id2label[p.item()] for p in predictions[0]
]

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, label in zip(tokens, predicted_labels):
    print(f"{token:15} → {label}")

[CLS]           → PUNCT
john            → PROPN
works           → VERB
at              → ADP
google          → PROPN
in              → ADP
california      → PROPN
[SEP]           → PUNCT


In [18]:
conll_label_names = conll_dataset["train"].features["chunk_tags"].feature.names

conll_label2id = {label: i for i, label in enumerate(conll_label_names)}
conll_id2label = {i: label for label, i in conll_label2id.items()}

print(conll_label_names)

['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


In [21]:
def tokenize_and_align_labels_chunk(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        else:
            # label is already an integer ID from the ClassLabel feature
            label = example["chunk_tags"][word_idx]
            labels.append(label)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [22]:
tokenized_conll = conll_dataset.map(tokenize_and_align_labels_chunk)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [23]:
tokenized_conll.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [24]:
num_labels = len(conll_label_names)

In [25]:
from transformers import AutoModelForTokenClassification

conll_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(conll_label_names),
    id2label={i: label for i, label in enumerate(conll_label_names)},
    label2id={label: i for i, label in enumerate(conll_label_names)},
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [28]:
import numpy as np

# Redefine compute_metrics for CoNLL task to use conll_id2label
def compute_metrics_conll(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, label in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for p_, l_ in zip(pred, label):
            if l_ != -100:
                # Use conll_id2label instead of the global id2label
                current_preds.append(conll_id2label[p_])
                current_labels.append(conll_id2label[l_])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

conll_trainer = Trainer(
    model=conll_model,
    args=training_args,
    train_dataset=tokenized_conll["train"],
    eval_dataset=tokenized_conll["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_conll, # Pass the new function
)

In [30]:
conll_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.107342,0.200363,0.922044,0.907303,0.914614
2,0.075358,0.206975,0.920743,0.915539,0.918134


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3512, training_loss=0.09260749526678294, metrics={'train_runtime': 819.544, 'train_samples_per_second': 34.265, 'train_steps_per_second': 4.285, 'total_flos': 593438990152956.0, 'train_loss': 0.09260749526678294, 'epoch': 2.0})

# 🔍 Comparison: POS Tagging vs Chunking

## 📌 Overview

In this assignment, we implemented two important token classification tasks using a transformer model:

- **Part-of-Speech (POS) Tagging**
- **Chunking (Phrase Detection)**

Both tasks assign labels to tokens, but they differ in complexity, purpose, and level of linguistic understanding.

---

## 🧠 1. Part-of-Speech (POS) Tagging

POS Tagging assigns a grammatical category to each word in a sentence.

### ✅ Characteristics:
- Works at **word-level**
- Identifies grammatical roles
- Easier compared to chunking

### 📌 Example:

Sentence:  
John works at Google  

POS Output:  
- John → PROPN  
- works → VERB  
- at → ADP  
- Google → PROPN  

---

## 🧠 2. Chunking (Phrase Detection)

Chunking groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP).

### ✅ Characteristics:
- Works at **phrase-level**
- Uses BIO tagging (B- = Begin, I- = Inside)
- More complex than POS tagging

### 📌 Example:

Sentence:  
John works at Google  

Chunking Output:  
- John → B-NP  
- works → B-VP  
- at → B-PP  
- Google → B-NP  

---

## ⚖️ Key Differences

- POS Tagging focuses on identifying the grammatical role of individual words.
- Chunking focuses on grouping words into meaningful phrases.
- POS operates at a simpler level, while chunking requires understanding of sequence and structure.

---

## 📊 Performance Comparison

- POS Tagging:
  - Precision: ~0.96  
  - Recall: ~0.96  
  - F1 Score: ~0.96  

- Chunking:
  - Precision: ~0.92  
  - Recall: ~0.91  
  - F1 Score: ~0.91  

---

## 📌 Observations

- POS tagging achieved higher performance because it is a simpler, word-level task.
- Chunking is more challenging as it requires learning relationships between tokens.
- The drop in F1 score reflects the increased complexity of phrase-level understanding.

---

## 🧠 Conclusion

POS tagging helps identify **what a word is**, while chunking helps identify **how words are grouped together** to form meaningful phrases.

Both tasks are essential for building advanced Natural Language Processing systems.

In [31]:
def tokenize_and_align_labels_ner(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        else:
            label = example["ner_tags"][word_idx]
            labels.append(label)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [32]:
tokenized_wikiann = wikiann_dataset.map(tokenize_and_align_labels_ner)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [33]:
tokenized_wikiann.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [34]:
from transformers import AutoModelForTokenClassification

wikiann_label_names = wikiann_dataset["train"].features["ner_tags"].feature.names

ner_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(wikiann_label_names),
    id2label={i: label for i, label in enumerate(wikiann_label_names)},
    label2id={label: i for i, label in enumerate(wikiann_label_names)},
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [35]:
ner_trainer = Trainer(
    model=ner_model,
    args=training_args,
    train_dataset=tokenized_wikiann["train"],
    eval_dataset=tokenized_wikiann["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [36]:
ner_trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.307389,0.278353,0.859780,0.874827,0.867238
2,0.203265,0.284714,0.874067,0.883757,0.878885


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: AUX seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: INTJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: 

TrainOutput(global_step=5000, training_loss=0.30226399536132814, metrics={'train_runtime': 937.4674, 'train_samples_per_second': 42.668, 'train_steps_per_second': 5.334, 'total_flos': 506975369554896.0, 'train_loss': 0.30226399536132814, 'epoch': 2.0})

In [37]:
import torch

sentence = "John works at Google in California"

tokens = sentence.split()

def predict(model, id2label):
    inputs = tokenizer(
        tokens,
        return_tensors="pt",
        is_split_into_words=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    predictions = torch.argmax(outputs.logits, dim=2)

    labels = [id2label[p.item()] for p in predictions[0]]
    tokens_out = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    return list(zip(tokens_out, labels))


print("🔹 POS TAGGING")
for t, l in predict(model, id2label):
    print(f"{t:15} → {l}")

print("\n🔹 CHUNKING")
for t, l in predict(conll_model, conll_id2label):
    print(f"{t:15} → {l}")

print("\n🔹 NER")
for t, l in predict(ner_model, {i: label for i, label in enumerate(wikiann_label_names)}):
    print(f"{t:15} → {l}")

🔹 POS TAGGING
[CLS]           → PUNCT
john            → PROPN
works           → VERB
at              → ADP
google          → PROPN
in              → ADP
california      → PROPN
[SEP]           → PUNCT

🔹 CHUNKING
[CLS]           → O
john            → B-NP
works           → B-VP
at              → B-PP
google          → B-NP
in              → B-PP
california      → B-NP
[SEP]           → O

🔹 NER
[CLS]           → O
john            → B-PER
works           → O
at              → O
google          → B-ORG
in              → O
california      → B-LOC
[SEP]           → I-PER


# 🧠 Final Conclusion: Token Classification using BERT

## 📌 Overview

In this assignment, a transformer-based model (BERT) was successfully fine-tuned for three different token classification tasks:

- Part-of-Speech (POS) Tagging  
- Chunking (Phrase Detection)  
- Named Entity Recognition (NER)  

A unified pipeline was implemented and reused across all tasks, demonstrating the flexibility of transformer models.

---

## 📊 Performance Summary

- POS Tagging:
  - F1 Score: ~0.96  
  - Highest performance due to simpler word-level classification  

- Chunking:
  - F1 Score: ~0.91  
  - Moderate complexity due to phrase-level dependencies  

- NER:
  - F1 Score: ~0.88  
  - Most challenging task requiring contextual understanding of entities  

---

## 🧠 Key Learnings

- Transformer models like BERT are highly effective for sequence labeling tasks.
- Tokenization and label alignment are critical steps, especially when dealing with subwords.
- A single pipeline can generalize across multiple NLP tasks with minimal changes.
- Performance decreases as task complexity increases from word-level to sequence-level understanding.

---

## ⚠️ Challenges Faced

- Handling subword tokenization and aligning labels correctly  
- Managing special tokens using `-100`  
- Dealing with dataset inconsistencies (string vs integer labels)  
- Resolving library version conflicts and API changes  
- Handling device mismatches (CPU vs GPU tensors)  

---

## 💡 Observations

- POS tagging achieved the highest accuracy as it focuses on individual tokens.
- Chunking required understanding of token relationships within phrases.
- NER required deeper contextual understanding, making it the most difficult task.
- Evaluation metrics like F1 score are more appropriate than accuracy for sequence labeling tasks.

---

## 🚀 Final Insight

This assignment demonstrates that transformer-based models can be effectively adapted to multiple NLP tasks using a unified approach.

The ability to reuse the same architecture across POS tagging, chunking, and NER highlights the power and flexibility of modern deep learning models in Natural Language Processing.

---

## 🎯 Conclusion

By completing this assignment, a complete token classification pipeline was built, covering:

- Data preprocessing  
- Tokenization and label alignment  
- Model fine-tuning  
- Evaluation and inference  
- Task comparison  

This provides a strong foundation for building advanced NLP applications such as chatbots, information extraction systems, and text understanding pipelines.